**dayofmonth() / day():**
- used to **extract** the **day of the month** (an integer between **1 and 31**) from a given **date or timestamp column**.
- Both **dayofmonth and day** functions accept a column of **date, timestamp, or a date string / timestamp string** data type.
- The functions return the **day of the month** as an **integer value between 1 and 31**.

##### Syntax

     dayofmonth(expr)

- `DateType default format is yyyy-MM-dd`. 
- `TimestampType default format is yyyy-MM-dd HH:mm:ss.SSSS`.
- Returns `null` if the input is a `string` that `can not be cast to Date or Timestamp`.
- The `default format` of the `Spark Timestamp` is `yyyy-MM-dd HH:mm:ss.SSSS`.

|  Argument  |  Description       |
|------------|--------------------|
| **expr**   | It accepts a column of **date or timestamp or a string** that can be implicitly **cast to a timestamp**. |
| **Output Type** | It returns the **day** component as an **integer**. |
| **Alternative** | You can also use the **shorter function** name **day()** which is an **alias** for **dayofmonth()**. |

     # Sample DataFrame                                
     df = spark.createDataFrame([('2015-04-08',)], ['dt'])

**dayofmonth**

     from pyspark.sql.functions import dayofmonth

     # Extract the day of the month
     df.select(dayofmonth('dt').alias('day_of_month')).display()

**day**

     from pyspark.sql.functions import day

     # Extract the day of the month using the 'day' function
     df.select(day('dt').alias('day_of_month')).display()

|  day_of_month  |
|----------------|
|     8          |

|  datetime_col 	|  day_of_month  |
|-----------------|----------------|
|   2024-10-31	  |       31       |
|   2015-04-08	  |        8       |
|   2026-01-29 10:00:00	| 29       |

In [0]:
from pyspark.sql.functions import col, lit, expr, dayofmonth, day, to_date, to_timestamp, date_format

##### 1) Using `dayofmonth` with a `string date Column`

In [0]:
data = [("2024-01-01",), ("2024-02-29",), ("2024-12-31",)]
df_02 = spark.createDataFrame(data, ["date"])
display(df_02)

date
2024-01-01
2024-02-29
2024-12-31


In [0]:
df_day_02 = df_02.select("date", expr("typeof(date)").alias("type"), dayofmonth("date").alias("day"))
display(df_day_02)

date,type,day
2024-01-01,string,1
2024-02-29,string,29
2024-12-31,string,31


##### 2) Using `dayofmonth` with a `date Column`

In [0]:
import datetime
df_03 = spark.createDataFrame([(datetime.date(2015, 4, 8),),
                               (datetime.date(2024, 10, 31),),
                               (datetime.date(2024, 12, 31),),
                               (datetime.date(2024, 1, 31),)], ['dt'])
display(df_03)

dt
2015-04-08
2024-10-31
2024-12-31
2024-01-31


In [0]:
df_03.select("*", expr("typeof(dt)").alias("type"), dayofmonth('dt')).display()

dt,type,dayofmonth(dt)
2015-04-08,date,8
2024-10-31,date,31
2024-12-31,date,31
2024-01-31,date,31


##### 3) Casting from String to a Date

In [0]:
df_01 = spark.createDataFrame([(1,)], ["id"]) \
    .withColumn("string", lit("2024-03-17")) \
    .withColumn("date", to_date(lit("2024-03-17")))

display(df_01)

id,string,date
1,2024-03-17,2024-03-17


In [0]:
df_day_01 = df_01.select("date", dayofmonth("date").alias("day"))
display(df_day_01)

date,day
2024-03-17,17


In [0]:
df_wmd_01 = df_01 \
    .withColumn('day_name', date_format('date', 'EEEE')) \
    .withColumn("day_of_month", dayofmonth(col("date")))

display(df_wmd_01)
df_wmd_01.printSchema()

id,date,day_name,day_of_month
1,2024-03-17,Sunday,17


root
 |-- id: long (nullable = true)
 |-- date: date (nullable = true)
 |-- day_name: string (nullable = true)
 |-- day_of_month: integer (nullable = true)



##### 4) Using `dayofmonth` with a `timestamp Column`

**a) dayofmonth with a string timestamp Column**

In [0]:
df_ts = spark.createDataFrame([("2024-03-17 14:30:00",)], ["timestamp"]) \
    .withColumn("ts", to_timestamp("timestamp"))

display(df_ts)

timestamp,ts
2024-03-17 14:30:00,2024-03-17T14:30:00.000Z


In [0]:
df_ts_01 = df_ts.select("ts", dayofmonth("ts").alias("day"))
display(df_ts_01)

ts,day
2024-03-17T14:30:00.000Z,17


**b) dayofmonth with a timestamp Column**

In [0]:
import datetime
df_ts_02 = spark.createDataFrame([(datetime.datetime(2015, 4, 8, 13, 8, 15),),
                                  (datetime.datetime(2024, 10, 31, 10, 9, 16),)], ['ts'])

display(df_ts_02)

ts
2015-04-08T13:08:15.000Z
2024-10-31T10:09:16.000Z


In [0]:
df_ts_02.select("*", expr("typeof(ts)").alias("type"), dayofmonth('ts').alias('month')).display()

ts,type,month
2015-04-08T13:08:15.000Z,timestamp,8
2024-10-31T10:09:16.000Z,timestamp,31


##### 5) Filtering Data
- `Get all records from 1st day of month`

In [0]:
# Sample table-like data
data = [
    ("2024-03-01", "Order-A"),
    ("2024-03-15", "Order-B"),
    ("2024-04-01", "Order-C"),
    ("2025-04-20", "Order-D"),
    ("2025-07-01", "Order-E"),
    ("2026-01-10", "Order-F"),
    ("2021-09-25", "Order-G"),
    ("2020-02-01", "Order-H"),
    ("2022-09-16", "Order-I"),
    ("2023-01-01", "Order-J")
]

# Create DataFrame
df_fltr = spark.createDataFrame(data, ["date", "order_id"]) \
    .withColumn("date", to_date("date", "yyyy-MM-dd"))

display(df_fltr)

date,order_id
2024-03-01,Order-A
2024-03-15,Order-B
2024-04-01,Order-C
2025-04-20,Order-D
2025-07-01,Order-E
2026-01-10,Order-F
2021-09-25,Order-G
2020-02-01,Order-H
2022-09-16,Order-I
2023-01-01,Order-J


In [0]:
df_fltr.filter(dayofmonth("date") == 1).display()

date,order_id
2024-03-01,Order-A
2024-04-01,Order-C
2025-07-01,Order-E
2020-02-01,Order-H
2023-01-01,Order-J


##### 6) Using dayofmonth in a GroupBy

In [0]:
# Sample data (spread across different days)
data = [
    ("2024-03-01", "Order-A"),
    ("2024-03-01", "Order-B"),
    ("2024-03-02", "Order-C"),
    ("2024-03-15", "Order-D"),
    ("2024-04-01", "Order-E"),
    ("2024-04-15", "Order-F"),
    ("2024-04-15", "Order-G"),
    ("2024-04-30", "Order-H"),
]

# Create DataFrame and cast date column
df_grp = spark.createDataFrame(data, ["date", "order_id"]) \
         .withColumn("date", to_date("date", "yyyy-MM-dd")) \
         .withColumn("day", dayofmonth("date"))

display(df_grp)

date,order_id,day
2024-03-01,Order-A,1
2024-03-01,Order-B,1
2024-03-02,Order-C,2
2024-03-15,Order-D,15
2024-04-01,Order-E,1
2024-04-15,Order-F,15
2024-04-15,Order-G,15
2024-04-30,Order-H,30


In [0]:
result = (
    df_grp.groupBy(dayofmonth("date").alias("day"))
          .count()
          .orderBy("day")
)

display(result)

day,count
1,3
2,1
15,3
30,1


##### 7) With Condition

In [0]:
# Sample data (mix of 1st and non-1st day)
data = [
    ("2024-03-01", "Order-A"),
    ("2024-03-02", "Order-B"),
    ("2024-04-01", "Order-C"),
    ("2024-04-15", "Order-D"),
    ("2024-05-01", "Order-E"),
    ("2024-05-20", "Order-F"),
]

# Create DataFrame and cast 'date' to DATE type
df_when = spark.createDataFrame(data, ["date", "order_id"]) \
         .withColumn("date", to_date("date", "yyyy-MM-dd"))

display(df_when)

date,order_id
2024-03-01,Order-A
2024-03-02,Order-B
2024-04-01,Order-C
2024-04-15,Order-D
2024-05-01,Order-E
2024-05-20,Order-F


In [0]:
from pyspark.sql.functions import when

# Add the is_month_start flag
df_when_final = df_when \
    .withColumn("day", dayofmonth("date")) \
    .withColumn("is_month_start", when(dayofmonth("date") == 1, "Yes").otherwise("No"))
display(df_when_final)

date,order_id,day,is_month_start
2024-03-01,Order-A,1,Yes
2024-03-02,Order-B,2,No
2024-04-01,Order-C,1,Yes
2024-04-15,Order-D,15,No
2024-05-01,Order-E,1,Yes
2024-05-20,Order-F,20,No
